<a href="https://colab.research.google.com/github/fdac25/MP2/blob/main/Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
import pandas as pd
import matplotlib.pyplot as plt



In [46]:
df = pd.read_csv("smurph61_project_summary.csv", sep=";")
df.columns = ['project','commit','author','time','message']
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df.head()

,project,commit,author,time,message,Month
0,easystats_parameters,0000296a2d6ee34cf9e717583df06a38e7842062,Daniel <mail@danielluedecke.de>,1684657123,Bug: `parameters` no longer respects the `vcov...,2023-05
1,easystats_parameters,000349ee3bbfe4b011a29a30f6d47a7a9486f5f6,GitHub Actions <actions@github.com>,1616069558,Built site for parameters: 0.12.0.1@14c1600\n,2021-03
2,easystats_parameters,00059db8ebf302fd696ae8a99df3b39ac57ea737,Daniel <mail@danielluedecke.de>,1603799266,save object name\n,2020-10
3,easystats_parameters,00084e7cfa0e749b92cca233453e6c5483ac5097,Daniel <mail@danielluedecke.de>,1600375007,fix check issues\n,2020-09
4,easystats_parameters,000b5bf7ed9e371fedc9625f78dd828d01f615d7,Daniel <mail@danielluedecke.de>,1659254294,lint\n,2022-07


In [47]:
projects = [
    'easystats_parameters',
    'ds4dm_tulip.jl',
    'mtiller_modelicabook',
    'jdtuck_fdasrvf_r',
    'xbarin02_collatz',
    'quantum-tii_qibo',
    'jklimke_libcitygml',
    'mskcc_vcf2maf',
    'sibrun_gh',
    'lyc102_ifem'
]

project_stats = []

for prj in projects:

    df1 = df[df['project'] == prj]

    df2 = df1.groupby('Month').size().reset_index(name='Commits')
    monthly_counts = df2
    monthly_counts['time'] = pd.to_datetime(monthly_counts['Month'])
    full_date_range = pd.DataFrame({'time': pd.date_range(start=monthly_counts['time'].min(),end=monthly_counts['time'].max(),freq='MS')})
    monthly_counts = pd.merge(full_date_range, monthly_counts, on='time', how='left').fillna(0)
    monthly_counts['Month'] = monthly_counts['time'].dt.strftime('%Y-%m')

    # Save monthly CSV
    monthly_counts[['Month', 'Commits']].rename(
        columns={'Commits': '#Commits'}
    ).to_csv(
        "smurph61_commits_timeseries_" + prj + ".csv",
        sep=";",
        index=False
    )
    zero_months = monthly_counts["Commits"] == 0
    
    gap_groups = (zero_months != zero_months.shift()).cumsum()
    
    gaps = monthly_counts[zero_months].groupby(gap_groups).agg(
        LongestGapStart=("Month", "first"),
        LongestGapEnd=("Month", "last"),
        LongestGapLength=("Month", "size")
    )
    
    if len(gaps) > 0:
        longest_gap = gaps.loc[gaps["LongestGapLength"].idxmax()]
    
        longest_gap_start = longest_gap["LongestGapStart"]
        longest_gap_end = longest_gap["LongestGapEnd"]
        longest_gap_length = int(longest_gap["LongestGapLength"])

        num_commits_after_gap = monthly_counts[monthly_counts["Month"] > longest_gap_end]["Commits"].sum()
    else:
        longest_gap_start = "N/A"
        longest_gap_end = "N/A"
        longest_gap_length = 0
        num_commits_after_gap = 0

    number_of_gaps = len(gaps[gaps["LongestGapLength"] >= 3])
    
    print(prj)
    print("Longest Gap Start:", longest_gap_start)
    print("Longest Gap End:", longest_gap_end)
    print("Longest Gap Length:", longest_gap_length)
    print("Commits After Gap:", int(num_commits_after_gap))
    print("Number of Gaps >= 3 Months:", number_of_gaps)
    print()

    # Create plot
    plt.figure(figsize=(10, 5))
    plt.plot(
        monthly_counts['Month'],
        monthly_counts['Commits'],
        marker='o',
        linestyle='-'
    )

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(prj)
    plt.grid(True)
    plt.xticks(rotation=45, fontsize=6)
    plt.tight_layout()

    plt.savefig("smurph61_timeseries_" + prj + ".png",dpi=300 )

    plt.close()

    project_stats.append({
        "Project": prj,
        "LongestGapStart": longest_gap_start,
        "LongestGapEnd": longest_gap_end,
        "LongestGapLength": longest_gap_length,
        "NumCommitsAfterLastGap": int(num_commits_after_gap),
        "NumberOfGapsInTimeline": number_of_gaps,
        "ActivityPattern": ""
    })
    

easystats_parameters
Longest Gap Start: N/A
Longest Gap End: N/A
Longest Gap Length: 0
Commits After Gap: 0
Number of Gaps >= 3 Months: 0

ds4dm_tulip.jl
Longest Gap Start: 2024-07
Longest Gap End: 2024-11
Longest Gap Length: 5
Commits After Gap: 32
Number of Gaps >= 3 Months: 3

mtiller_modelicabook
Longest Gap Start: 2022-01
Longest Gap End: 2022-09
Longest Gap Length: 9
Commits After Gap: 144
Number of Gaps >= 3 Months: 12

jdtuck_fdasrvf_r
Longest Gap Start: 2014-08
Longest Gap End: 2015-10
Longest Gap Length: 15
Commits After Gap: 847
Number of Gaps >= 3 Months: 10

xbarin02_collatz
Longest Gap Start: 2024-04
Longest Gap End: 2024-11
Longest Gap Length: 8
Commits After Gap: 56
Number of Gaps >= 3 Months: 5

quantum-tii_qibo
Longest Gap Start: N/A
Longest Gap End: N/A
Longest Gap Length: 0
Commits After Gap: 0
Number of Gaps >= 3 Months: 0

jklimke_libcitygml
Longest Gap Start: 2020-12
Longest Gap End: 2021-09
Longest Gap Length: 10
Commits After Gap: 305
Number of Gaps >= 3 Months

In [48]:
activity_patterns = {
    'easystats_parameters': 'declining',
    'ds4dm_tulip.jl': 'declining',
    'mtiller_modelicabook': 'declining',
    'jdtuck_fdasrvf_r': 'irregular',
    'xbarin02_collatz': 'declining',
    'quantum-tii_qibo': 'irregular',
    'jklimke_libcitygml': 'U-shaped',
    'mskcc_vcf2maf': 'declining',
    'sibrun_gh': 'irregular',
    'lyc102_ifem': 'declining'
}

In [49]:
github_stats = {
    'easystats_parameters': [499, 46, '2026-10-05'],
    'ds4dm_tulip.jl': [163, 24, '2026-04-19'],
    'mtiller_modelicabook': [117, 75, '2026-09-06'],
    'jdtuck_fdasrvf_r': [15, 18, '2026-09-30'],
    'xbarin02_collatz': [84, 7, '2026-10-05'],
    'quantum-tii_qibo': [366, 104, '2026-10-07'],
    'jklimke_libcitygml': [111, 59, '2025-11-25'],
    'mskcc_vcf2maf': [421, 219, '2026-05-25'],
    'sibrun_gh': [2, 3, '2026-03-11'],
    'lyc102_ifem': [297, 99, '2026-08-19']
}


In [50]:
stats_df = pd.DataFrame(project_stats)

for i, row in stats_df.iterrows():
    prj = row["Project"]
    project_df = df[df["project"] == prj]

    stats_df.loc[i, "ncommits"] = project_df["commit"].nunique()
    stats_df.loc[i, "nauthors"] = project_df["author"].nunique()
    stats_df.loc[i, "from"] = project_df["time"].min()
    stats_df.loc[i, "to"] = project_df["time"].max()

    stats_df.loc[i, "nstars"] = github_stats[prj][0]
    stats_df.loc[i, "nforks"] = github_stats[prj][1]
    stats_df.loc[i, "lastGHCommitDate"] = github_stats[prj][2]

    stats_df.loc[i, "ActivityPattern"] = activity_patterns[prj]

stats_df = stats_df[
    [
        "Project",
        "ncommits",
        "nauthors",
        "from",
        "to",
        "nstars",
        "nforks",
        "lastGHCommitDate",
        "LongestGapStart",
        "LongestGapEnd",
        "LongestGapLength",
        "NumCommitsAfterLastGap",
        "ActivityPattern",
        "NumberOfGapsInTimeline"
    ]
]

stats_df.to_csv(
    "smurph61_project_stats.csv",
    index=False
)

stats_df

,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,ActivityPattern,NumberOfGapsInTimeline
0,easystats_parameters,8925.0,76.0,1.549685e+09,1.776170e+09,499.0,46.0,2026-10-05,N/A,N/A,0,0,declining,0
1,ds4dm_tulip.jl,926.0,29.0,1.524835e+09,1.759339e+09,163.0,24.0,2026-04-19,2024-07,2024-11,5,32,declining,3
2,mtiller_modelicabook,2558.0,69.0,1.354808e+09,1.742474e+09,117.0,75.0,2026-09-06,2022-01,2022-09,9,144,declining,12
3,jdtuck_fdasrvf_r,891.0,25.0,1.347235e+09,1.760872e+09,15.0,18.0,2026-09-30,2014-08,2015-10,15,847,irregular,10
4,xbarin02_collatz,1462.0,4.0,1.564771e+09,1.762346e+09,84.0,7.0,2026-10-05,2024-04,2024-11,8,56,declining,5
5,quantum-tii_qibo,14496.0,137.0,1.582014e+09,1.762324e+09,366.0,104.0,2026-10-07,N/A,N/A,0,0,irregular,0
6,jklimke_libcitygml,598.0,58.0,1.349084e+09,1.760909e+09,111.0,59.0,2025-11-25,2020-12,2021-09,10,305,U-shaped,9
7,mskcc_vcf2maf,498.0,42.0,1.385144e+09,1.754097e+09,421.0,219.0,2026-05-25,2021-05,2022-05,13,36,declining,5
8,sibrun_gh,1032.0,11.0,1.511975e+09,1.759783e+09,2.0,3.0,2026-03-11,2018-08,2021-03,32,700,irregular,5
9,lyc102_ifem,359.0,14.0,1.559805e+09,1.760729e+09,297.0,99.0,2026-08-19,2023-03,2023-10,8,22,declining,4


# Interpretation

### easystats_parameters
The activity pattern is declining. Commit activity is high in the earlier years but generally decreases over time. The project has 0 inactivity gaps of at least three months.

### ds4dm_tulip.jl
The activity pattern is declining. The project has much higher activity early in its history, followed by lower and less frequent activity in later years. Its longest inactivity gap occurred from 2024-07 to 2024-11 and lasted 5 months. The project has 3 gaps of at least three months.

### mtiller_modelicabook
The activity pattern is declining. The project has frequent bursts of activity in early year, while later years have significantly less spikes. Its longest inactivity gap occurred from 2022-01 to 2022-09 and lasted 9 months. The project has 12 gaps of at least three months.

### jdtuck_fdasrvf_r
The activity pattern is irregular. Development occurs in uneven bursts separated by periods of little or no activity. It has no consistent upward or downward trend. Its longest inactivity gap occurred from 2014-08 to 2015-10 and lasted 15 months. The project has 10 gaps of at least three months.

### xbarin02_collatz
The activity pattern is declining. The project begins with extremely high commit activity. Activity declines sharply and remains low for most of the later timeline. Its longest inactivity gap occurred from 2024-04 to 2024-11 and lasted 8 months. The project has 5 gaps of at least three months.

### quantum-tii_qibo
The activity pattern is irregular. The project remains active throughout the timeline, however commit counts are not consistant can contain many spikes. There are no inactivity gaps in the monthly timeline, so the number of gaps of at least three months is 0.

### jklimke_libcitygml
The activity pattern is U-shaped. Activity is high early in the timeline, declines to very low levels through the middle period, and then increases again toward the end. Its longest inactivity gap occurred from 2020-12 to 2021-09 and lasted 10 months. The project has 9 gaps of at least three months.

### mskcc_vcf2maf
The activity pattern is declining. Earlier years show consistent commit activity. Later years show  inactive periods and only occasional small bursts. Its longest inactivity gap occurred from 2021-05 to 2022-05 and lasted 13 months. The project has 5 gaps of at least three months.

### sibrun_gh
The activity pattern is irregular. The project contains several large bursts of commits with long periods with little or no activity in between. Its longest inactivity gap occurred from 2018-08 to 2021-03 and lasted 32 months. The project has 5 gaps of at least three months.

### lyc102_ifem
The activity pattern is declining. The highest levels of activity occur early in the project, followed by consistently low activity. Its longest inactivity gap occurred from 2023-03 to 2023-10 and lasted 8 months. The project has 4 gaps of at least three months.
